## Gold Layer — Titanic Star Schema

One fact and four dimensions, deliberately kept lightweight so the shape is recognisable in any BI tool.

```
                        dim_passenger  ──┐
                                         │
   dim_class  ── FactPassengerVoyage ── dim_port
                                         │
                        dim_date  ───────┘   (demo dimension; Titanic only has one date)
```

**Keys:** surrogate integer keys generated with `monotonically_increasing_id`. For a teaching demo this is fine;
in a production warehouse use `ROW_NUMBER()` over a stable sort, or a sequence/identity column.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_table = "silver_passenger"
df_silver = spark.table(silver_table)

In [ ]:
# --- dim_passenger ----------------------------------------------------------
# Passenger is the degenerate dimension here: one row per person, no SCD needed.
dim_passenger = (
    df_silver.select(
        "PassengerId", "Name", "Title", "Sex", "Age", "AgeGroup",
        "SiblingsSpousesAboard", "ParentsChildrenAboard",
        "FamilySize", "IsAlone", "CabinDeck",
    )
    .withColumn(
        "AgeGroupSort",
        F.when(F.col("AgeGroup") == "Child",       1)
         .when(F.col("AgeGroup") == "Teen",        2)
         .when(F.col("AgeGroup") == "Adult",       3)
         .when(F.col("AgeGroup") == "MiddleAged",  4)
         .when(F.col("AgeGroup") == "Senior",      5)
         .otherwise(99)
    )
    .withColumn(
        "PassengerKey",
        F.row_number().over(Window.orderBy("PassengerId"))
    )
    .select(
        "PassengerKey", "PassengerId", "Name", "Title", "Sex",
        "Age", "AgeGroup", "AgeGroupSort",
        "SiblingsSpousesAboard", "ParentsChildrenAboard",
        "FamilySize", "IsAlone", "CabinDeck",
    )
)

In [ ]:
# --- dim_class --------------------------------------------------------------
dim_class = (
    df_silver.select(
        F.col("PassengerClassNumber").alias("ClassNumber"),
        F.col("PassengerClass").alias("ClassName"),
    )
    .distinct()
    .withColumn("ClassDescription",
        F.when(F.col("ClassNumber") == 1, "First class — upper deck, luxury cabins")
         .when(F.col("ClassNumber") == 2, "Second class — middle tier")
         .when(F.col("ClassNumber") == 3, "Third class — steerage, lower decks")
         .otherwise("Unknown")
    )
    .withColumn("ClassKey", F.row_number().over(Window.orderBy("ClassNumber")))
    .select("ClassKey", "ClassNumber", "ClassName", "ClassDescription")
)

In [ ]:
# --- dim_port ---------------------------------------------------------------
port_meta = spark.createDataFrame(
    [
        ("Southampton", "S", "England",         50.9097,  -1.4044),
        ("Cherbourg",   "C", "France",          49.6337,  -1.6222),
        ("Queenstown",  "Q", "Ireland (Cobh)",  51.8500,  -8.3000),
        ("Unknown",     "U", "Unknown",         None,     None),
    ],
    ["PortName", "PortCode", "Country", "Latitude", "Longitude"],
)

dim_port = (
    port_meta.withColumn("PortKey", F.row_number().over(Window.orderBy("PortCode")))
             .select("PortKey", "PortName", "PortCode", "Country", "Latitude", "Longitude")
)

In [ ]:
# --- dim_date ---------------------------------------------------------------
# Titanic has one voyage, but a date dim is where a real project would attach departure/arrival.
# Here: one row for the disaster date so the model is still conformant.
dim_date = spark.createDataFrame(
    [("1912-04-15",)],
    ["FullDate"],
).select(
    F.lit(1).alias("DateKey"),
    F.to_date("FullDate").alias("FullDate"),
    F.year(F.to_date("FullDate")).alias("Year"),
    F.quarter(F.to_date("FullDate")).alias("Quarter"),
    F.month(F.to_date("FullDate")).alias("Month"),
    F.date_format(F.to_date("FullDate"), "MMMM").alias("MonthName"),
    F.day(F.to_date("FullDate")).alias("DayOfMonth"),
    F.date_format(F.to_date("FullDate"), "EEEE").alias("DayName"),
    F.lit("Titanic sinking").alias("EventDescription"),
)

In [ ]:
# --- fact_passenger_voyage --------------------------------------------------
# Grain: one row per passenger per voyage (Titanic has one voyage, so effectively one row per passenger).
# Measures: Survived (0/1 — additive count), Fare (additive), FarePerPerson (semi-additive — avg only).
fact = (
    df_silver.alias("s")
    .join(dim_passenger.alias("p"), F.col("s.PassengerId") == F.col("p.PassengerId"), "inner")
    .join(dim_class.alias("c"),     F.col("s.PassengerClassNumber") == F.col("c.ClassNumber"), "inner")
    .join(dim_port.alias("po"),     F.col("s.EmbarkedPort") == F.col("po.PortName"), "left")
    .join(dim_date.alias("d"),      F.lit(1) == F.col("d.DateKey"), "inner")
    .select(
        F.col("p.PassengerKey"),
        F.col("c.ClassKey"),
        F.coalesce(F.col("po.PortKey"),
                   F.lit(dim_port.filter("PortName = 'Unknown'").select("PortKey").first()[0])
                  ).alias("PortKey"),
        F.col("d.DateKey"),
        F.col("s.Ticket").alias("TicketNumber"),
        F.col("s.Cabin").alias("CabinNumber"),
        F.col("s.Fare"),
        (F.col("s.Fare") / F.col("s.FamilySize")).alias("FarePerPerson"),
        F.col("s.Survived").cast("int").alias("SurvivedFlag"),
        F.lit(1).cast("int").alias("PassengerCount"),
    )
)

In [ ]:
# --- Write out --------------------------------------------------------------
def write_delta(df, name):
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(name))
    print(f"  {name:35s}  rows: {spark.table(name).count()}")

print("Writing Gold model:")
write_delta(dim_passenger, "gold_dim_passenger")
write_delta(dim_class,     "gold_dim_class")
write_delta(dim_port,      "gold_dim_port")
write_delta(dim_date,      "gold_dim_date")
write_delta(fact,          "gold_fact_passenger_voyage")

In [ ]:
# --- Sanity check: a classic Titanic query via the star -----------------------
spark.sql("""
    SELECT
        c.ClassName,
        p.Sex,
        COUNT(*)                            AS Passengers,
        SUM(f.SurvivedFlag)                 AS Survivors,
        ROUND(AVG(f.SurvivedFlag) * 100, 1) AS SurvivalRatePct,
        ROUND(AVG(f.Fare), 2)               AS AvgFare
    FROM gold_fact_passenger_voyage f
    JOIN gold_dim_passenger p ON f.PassengerKey = p.PassengerKey
    JOIN gold_dim_class     c ON f.ClassKey     = c.ClassKey
    GROUP BY c.ClassName, p.Sex
    ORDER BY c.ClassName, p.Sex
""").show()